# 03.3 — Cleaning and normalization

Every parser in the last two notebooks returned an empty string for one document
and said nothing about it.

This notebook is about catching that before it reaches your index, and about the
other things that arrive broken: text that OCR mangled, characters that got
mis-decoded, and boilerplate that appears in every chunk you own.

In [1]:
!pip install -q pymupdf4llm==1.28.2 pytesseract==0.3.13 pdf2image==1.17.0

## System packages

Two of the things this notebook needs are not Python packages and `pip` cannot
install them:

- **Tesseract** — the OCR engine `pytesseract` drives
- **Poppler** — the PDF tools `pdf2image` shells out to

The cell below installs them and then checks whether it worked. It does not hide
the output, because an install that fails quietly is how you end up debugging
`PDFInfoNotInstalledError` twenty minutes later.

In [2]:
import shutil, subprocess, os

NEEDED = {'tesseract': 'tesseract-ocr', 'pdftoppm': 'poppler-utils'}
missing = {b: pkg for b, pkg in NEEDED.items() if not shutil.which(b)}

if missing:
    sudo = [] if os.geteuid() == 0 else ['sudo']
    subprocess.run([*sudo, 'apt-get', 'update', '-qq'])
    subprocess.run([*sudo, 'apt-get', 'install', '-y', '-qq', *sorted(set(missing.values()))])

for binary, package in NEEDED.items():
    path = shutil.which(binary)
    print(f'{binary:<12} {path or f"NOT FOUND — install {package}"}')

tesseract    /usr/bin/tesseract
pdftoppm     /usr/bin/pdftoppm


If either line says NOT FOUND, the OCR section below will fail and the rest of the
notebook will still run.

On a machine where you can't `apt-get` — a locked-down laptop, some Kaggle
configurations — read the OCR results in the markdown and carry on. Nothing after
that section depends on it.

## Triage: look before you parse

The instinct is to call a parser and check whether the result looks empty. That's
backwards — by then you've already spent the time, and "looks empty" is a
judgement you'd have to make on every document forever.

Inspect first, then route. For PDFs the question is whether there's a text layer
at all, and the reliable test is **embedded fonts**, not extracted characters.

In [3]:
import pymupdf
from pathlib import Path

CORPUS = Path('../../corpus/docs')

for path in sorted(CORPUS.glob('*.pdf')):
    with pymupdf.open(path) as doc:
        fonts = sum(len(page.get_fonts()) for page in doc)
        chars = sum(len(page.get_text().strip()) for page in doc)
    flag = '  <-- no text layer' if fonts == 0 else ''
    print(f'fonts={fonts:>3}  chars={chars:>6} {path.name}{flag}')

fonts=  7  chars=  5843 kaduna-agro-annual-report-2024.pdf
fonts=  5  chars=  3857 kaduna-agro-board-minutes-2024-10-17.pdf
fonts=  0  chars=     0 kdirs-guidance-note-4-2024-SCANNED.pdf  <-- no text layer
fonts=  4  chars=  4117 nfsc-circular-2024-07-cybersecurity.pdf
fonts=  4  chars=  2183 nfsc-circular-2025-02-amendment.pdf
fonts=  7  chars=  6275 sahel-employee-handbook-2023.pdf
fonts=  7  chars=  6364 sahel-employee-handbook-2025.pdf
fonts=  3  chars=  3445 sahel-procurement-policy-v3.pdf


Zero fonts, zero characters. Everything else has three to seven fonts.

Why fonts rather than characters? A scan can still yield a few stray characters —
a digital stamp, a signature overlay, an invisible OCR layer somebody added years
ago. Checking `len(text) > 0` would pass those and you'd index a page of noise.
Fonts are the honest signal: no embedded font means nothing was ever set as text.

So we have a routing rule:

In [4]:
def inspect_pdf(path):
    with pymupdf.open(path) as doc:
        fonts = sum(len(page.get_fonts()) for page in doc)
        pages = doc.page_count
    return {'pages': pages, 'has_text_layer': fonts > 0, 'route': 'text' if fonts else 'ocr'}

scan = CORPUS / 'kdirs-guidance-note-4-2024-SCANNED.pdf'
print(inspect_pdf(scan))
print(inspect_pdf(CORPUS / 'sahel-procurement-policy-v3.pdf'))

{'pages': 1, 'has_text_layer': False, 'route': 'ocr'}
{'pages': 1, 'has_text_layer': True, 'route': 'text'}


## OCR, and what it costs you

The scan routes to OCR. Run it and read the output carefully — this is not a
solved problem.

In [5]:
import pytesseract
from pdf2image import convert_from_path

page = convert_from_path(scan, dpi=200)[0]
text = pytesseract.image_to_string(page)

print(f'{len(text)} characters recovered\n')
print(text[:420])

1726 characters recovered

Kaduna State Internal Revenue Service
Guidarice Note 4 of 2024 — Tax Clearance Certificates for Corporate Vendors

Issued: 9 September 2024 + Bivex torate of Assessment
1. Application

the purpose of tendering for contracts with State Ministries,
tax compliance as a condition of vendor registration.

2. Documents required

An application shall be accompanied by the certificate of incorporation, the memorandi
Statemen


Two different kinds of damage in there, and the second is much worse than the
first.

**Garbled words.** *Guidance* became *Guidarice*. *Directorate of Assessment*
became *Bivex torate of Assessment*. A pipe character became a plus. Annoying,
and mostly survivable — an embedding model will still put a chunk containing
*Guidarice Note 4 of 2024* somewhere near a question about guidance notes.

**Missing text.** Look at section 1. In the original it opens with a sentence
about which assessment years the note applies to. Here that sentence is simply
gone, and the line that follows it is cut off partway through.

Nothing marks the gap. The output reads as continuous prose. A chunk built from
this looks perfectly reasonable and is missing a clause that changes its meaning.

## The settings are a trade-off, not a quality dial

The obvious response is to turn the resolution up. Try it.

In [6]:
attempts = {
    '200 dpi, default': (200, ''),
    '400 dpi, psm 6':   (400, '--psm 6'),
}

probes = ['fourteen working days', '31 December', 'ninety days', 'Validity']

results = {}
for label, (dpi, config) in attempts.items():
    img = convert_from_path(scan, dpi=dpi)[0]
    results[label] = pytesseract.image_to_string(img, config=config)

print(f"{'pharse':<26}" + ''.join(f'{k:<20}' for k in results))
print('-' * 66)
for probe in probes:
    row = f'{probe:<26}'
    for text in results.values():
        row += f"{'found' if probe.lower() in text.lower() else 'MISSING':<20}"
    print(row)

print()
for label, text in results.items():
    print(f'{label:<20} {len(text):>6,} characters')

pharse                    200 dpi, default    400 dpi, psm 6      
------------------------------------------------------------------
fourteen working days     MISSING             found               
31 December               found               MISSING             
ninety days               found               found               
Validity                  found               found               

200 dpi, default      1,726 characters
400 dpi, psm 6        2,747 characters


Raising the resolution recovered about sixty per cent more text — and lost
*31 December*.

That phrase is the answer to one of the golden questions about this document:
how long a Tax Clearance Certificate is valid. The higher-resolution pass reads
more of the page and misreads that particular line.

So the setting that produces more text produces a worse result for the question
somebody will actually ask.

You could not have discovered that by looking at the output. Both versions read
fine. **OCR configuration is something you tune against known answers or not at
all** — which is the same argument this course makes about chunking, embeddings
and everything else, arriving early and from an unexpected direction.

For our corpus we keep 200 dpi, because it retains the answer we care about. On
a different corpus that would be the wrong call.

## Encoding damage

The other way text arrives broken, and the one that's easiest to fix once you can
recognise it.

A file written as UTF-8 and read as ISO-8859-1 doesn't fail. It succeeds, and
gives you nonsense.

In [7]:
original = 'The Bank\u2019s policy \u2014 effective 1 January \u2014 applies to all staff.'
print('as written: ', original)

damaged = original.encode('utf-8').decode('iso-8859-1')
print('misdecoded:', damaged)

as written:  The Bank’s policy — effective 1 January — applies to all staff.
misdecoded: The Bankâs policy â effective 1 January â applies to all staff.


Those garbled sequences are a misread apostrophe and a misread dash. This is
called mojibake, and it turns up constantly in exports from older systems.

It matters more than it looks. Those sequences become tokens, they land in your
embeddings, and a chunk full of them sits slightly further from every question
than it should. It degrades retrieval quietly rather than breaking it loudly.

Two defences. Detect the encoding rather than assuming, and repair what slips
through:

In [8]:
def repair_mojibake(text):
    """Undo a UTF-8 -> ISO-8859-1 misdecode, if that's what happened."""
    if 'â€' not in text and 'Ã' not in text:
        return text
    try:
        return text.encode('iso-8859-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text # not this failure mode; leave it alone

print(repair_mojibake(damaged))
print(repair_mojibake('nothing wrong here'))

The Bankâs policy â effective 1 January â applies to all staff.
nothing wrong here


The guard matters. Applying that transformation to text that *isn't* mojibake
corrupts it, so the function checks for the signature first and returns the input
unchanged when it doesn't match.

For detection rather than repair, `charset-normalizer` will tell you what a file
probably is instead of you guessing.

## Boilerplate

Every document here carries a footer saying it's synthetic. Yours will carry a
confidentiality notice, a page number, a copyright line, or all three.

The obvious fix is to drop lines that repeat across documents. Look at the footer
before writing that code.

In [9]:
import pymupdf4llm

text = pymupdf4llm.to_markdown(str(CORPUS / 'sahel-employee-handbook-2025.pdf'))

for line in text.splitlines():
    if 'SYNTHETIC' in line:
        print(repr(line.strip()))


=== Document parser messages ===
Using Tesseract for OCR processing.
'~~This handbook is reviewed every two years by the Head of Human Capital and approved by the Board Governance Committee. Interim~~ SYNTHETIC DOCUMENT — generated for the RAG Engineering course. Fictional organisation. Not a real filing. Do not cite as fact. amendments are circulated by internal memorandum and take effect from the date stated in the memorandum.'


Read that carefully. It is not a footer on its own line.

The real sentence is *"This handbook is reviewed every two years by the Head of
Human Capital and approved by the Board Governance Committee. Interim amendments
are circulated by internal memorandum..."*

The footer has been inserted **between `Interim` and `amendments`**.

That isn't a bug in the parser. The footer sits at the bottom of the page, the
parser reads the page by position, and the paragraph wraps around it. Any
position-based reader does this.

There's a second artifact. The text before the footer is wrapped in `~~` —
markdown for strikethrough. The parser saw the footer's horizontal rule and
concluded the line above it was struck through, so a perfectly normal paragraph
now claims to be deleted text.

**So the obvious fix destroys content.** Drop that line and a policy sentence goes
with it.

In [10]:
line = next(l for l in text.splitlines() if 'SYNTHETIC' in l)

print('deleting that line would also remove:\n')
print(' ', line.split('SYNTHETIC')[0].strip()[:130], '...')
print(' ...', line.split('cite as fact.')[-1].strip()[:130])

deleting that line would also remove:

  ~~This handbook is reviewed every two years by the Head of Human Capital and approved by the Board Governance Committee. Interim~~ ...
 ... amendments are circulated by internal memorandum and take effect from the date stated in the memorandum.


Remove the boilerplate as a **substring** instead, and clear up the strikethrough
markers it left behind.

In [11]:
import re

BOILERPLATE = re.compile(
    r'SYNTHETIC DOCUMENT.*?(?:cite as fact\.|Not a real record\.)', re.DOTALL)

def strip_boilerplate(text):
    text = BOILERPLATE.sub('', text)
    text = text.replace('~~', '') # spurious strikethrough around the footer
    return re.sub(r'  +', ' ', text)

cleaned = strip_boilerplate(text)

i = cleaned.find('This handbook is reviewed')
print(repr(cleaned[i:i +250]))
print(f"\nboilerplate gone: {'SYNTHETIC' not in cleaned}")
print(f'characters removed: {len(text) - len(cleaned):,}')

'This handbook is reviewed every two years by the Head of Human Capital and approved by the Board Governance Committee. Interim amendments are circulated by internal memorandum and take effect from the date stated in the memorandum. \n\n'

boilerplate gone: True
characters removed: 137


The sentence is whole again and the footer is gone.

The general lesson is worth more than the regex. **Boilerplate is not reliably on
its own line**, so line-based removal is dangerous — it deletes real content
silently, and you find out months later when questions about handbook amendments
stop working.

Before writing any removal rule, print the thing you're about to remove and look
at what it's attached to.

## Putting it together

In [12]:
def clean(text):
    text = repair_mojibake(text)
    text = strip_boilerplate(text)

    out, blank = [], False
    for line in text.splitlines():
        if line.strip():
            out.append(line.rstrip())
            blank = False
        elif not blank:
            out.append('')
            blank = True
    return '\n'.join(out).strip()

for name in ['sahel-employee-handbook-2025.pdf', 'kaduna-agro-annual-report-2024.pdf']:
    raw = pymupdf4llm.to_markdown(str(CORPUS / name))
    print(f'{name}\n  {len(raw):>7} -> {len(clean(raw)):>7,} chars')


=== Document parser messages ===
Using Tesseract for OCR processing.
sahel-employee-handbook-2025.pdf
     6559 ->   6,387 chars

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
kaduna-agro-annual-report-2024.pdf
     6352 ->   6,185 chars


## What's next

The scan is readable, the encoding is repaired, the boilerplate is gone.

But something is still missing from the OCR'd document, and from every other one:
it has no idea what it is. There's nothing recording that it came from a revenue
service, that it was issued in September 2024, or that it was recovered by OCR
and should be trusted less than the rest.

Notebook 4 is about structure — what gets destroyed when you flatten a document
to plain text. Notebook 5 is about the metadata that has to survive alongside it,
and it's the notebook that fixes the failure that cost you seven questions in
module 02.